In [2]:
import pandas as pd

df = pd.read_csv("triagem-avaliacoes/data/olist_order_reviews_dataset.csv")
print(df.shape)
df.head()

(99224, 7)


,review_id,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp
0,7bc2406110b926393aa56f80a40eba40,73fc7af87114b39712e6da79b0a377eb,4,NaN,NaN,2018-01-18 00:00:00,2018-01-18 21:46:59
1,80e641a11e56f04c1ad469d5645fdfde,a548910a1c6147796b98fdf73dbeba33,5,NaN,NaN,2018-03-10 00:00:00,2018-03-11 03:05:13
2,228ce5500dc1d8e020d8d1322874b6f0,f9e4b658b201a9f2ecdecbb34bed034b,5,NaN,NaN,2018-02-17 00:00:00,2018-02-18 14:36:24
3,e64fb393e7b32834bb789ff8bb30750e,658677c97b385a9be170737859d3511b,5,NaN,Recebi bem antes do prazo estipulado.,2017-04-21 00:00:00,2017-04-21 22:02:06
4,f7c4243c7fe1938f181bec41a392bdeb,8e6bfb81e283fa7e4f11123a3fb894f1,5,NaN,Parabéns lojas lannister adorei comprar pela I...,2018-03-01 00:00:00,2018-03-02 10:26:53


In [3]:
# Quantas avaliações têm texto?
com_texto = df.dropna(subset=["review_comment_message"]).copy()
print(f"{len(com_texto)} de {len(df)} têm comentário ({len(com_texto)/len(df):.0%})")

# Distribuição das notas (1 a 5)
print(com_texto["review_score"].value_counts().sort_index())

40977 de 99224 têm comentário (41%)
review_score
1     8745
2     2145
3     3557
4     5976
5    20554
Name: count, dtype: int64


In [4]:
# Tamanho dos comentários, em palavras
com_texto["n_palavras"] = com_texto["review_comment_message"].str.split().str.len()
print(com_texto["n_palavras"].describe())

count    40977.000000
mean        11.679210
std          9.554347
min          0.000000
25%          4.000000
50%          9.000000
75%         16.000000
max         45.000000
Name: n_palavras, dtype: float64


In [5]:
# Ler exemplos reais de notas baixas: é daqui que saem suas categorias
pd.set_option("display.max_colwidth", 200)
com_texto[com_texto["review_score"] <= 2]["review_comment_message"].sample(30, random_state=50)

2351                                                                                                     Comprei uma coisa e me entregaram outra e não consigo fazer a troca e não consigo falar com a central !
15383    Estou aguardando,espero que entreguem,pois o período das festas acabaram,sou otimista,nunca tive problemas com essa loja,tudo que comprei foi entregue,mas nunca havia adquirido produtos no final d...
98457                                                                    Uma empresa tão bem conceituada como o baratheon deveria se preocupar mais com a qualidade dos produtos oferecidos pelo seus parceiros.
32086                                    Só foi entregue o cartucho da tinta preta. Falta o outro cartucho da tinta colorida. Entretanto, a cobrança foi dos 2 cartuchos e ainda foi duplicado o valor do frete.
79332                                                                                                                                                               

In [6]:
import re

com_texto["texto"] = (com_texto["review_comment_message"]
                      .str.replace(r"[\r\n]+", " ", regex=True)
                      .str.strip())
com_texto = com_texto[com_texto["texto"].str.len() > 2]
com_texto = com_texto.drop_duplicates(subset="texto")
print(len(com_texto))

35565
